# Obstacle-aware coverage benchmark — runner (benchmark_v1_2, f2c_bench 1.4)

This notebook runs `f2c_bench.py`, a headless batch runner that plans strategies **A, B1, B2 and B3** with Fields2Cover over the factorial design.
It writes one CSV row per (instance, strategy, SC) and GeoJSON/coverage dumps for the seed-0 instances.

* Results go to **Google Drive** (`MyDrive/F2C_benchmark/benchmark_v1_2`), so they survive a disconnect. **Re-run the "run" cell to resume**: finished instances are skipped.
* Each instance is planned in its own subprocess, so a crash inside Fields2Cover only loses that instance (see `parts/*.err`).
* Expected cost is ≈2.5k route plans at a 10 s OR-Tools budget, i.e. **≈5–6 h with 2 workers**. Colab may disconnect, so plan on 1–2 resumes.

The same `.py` files also run headless on a Linux VM (see the last cell).

In [ ]:
# ---- 1. Install Fields2Cover (same recipe as the tutorial, idempotent) ----
import importlib.util, os, subprocess, time, glob
ORTOOLS_URL = ("https://github.com/google/or-tools/releases/download/v9.9/"
               "or-tools_amd64_ubuntu-22.04_cpp_v9.9.3963.tar.gz")
def sh(cmd, env=None):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env={**os.environ, **(env or {})})
    if r.returncode: print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(cmd)
if not os.path.exists("/opt/or-tools/lib/cmake/ortools"):
    sh(f"wget -q -O /tmp/ortools.tgz {ORTOOLS_URL} && mkdir -p /opt/or-tools && "
       "tar -xzf /tmp/ortools.tgz -C /opt/or-tools --strip-components=1 && "
       "echo /opt/or-tools/lib > /etc/ld.so.conf.d/ortools.conf && ldconfig")
if importlib.util.find_spec("fields2cover") is None:
    t0 = time.time()
    sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y build-essential cmake git "
       "libgdal-dev libgeos-dev libeigen3-dev libtbb-dev libtinyxml2-dev nlohmann-json3-dev libboost-dev")
    wheels = sorted(glob.glob("/content/drive/MyDrive/F2C_cache/fields2cover-*.whl"))
    if not wheels:
        sh("pip wheel --no-deps -w /content/f2c_wheels 'git+https://github.com/Fields2Cover/Fields2Cover.git@v2.1.1'",
           env={"CMAKE_PREFIX_PATH": "/opt/or-tools", "CMAKE_BUILD_PARALLEL_LEVEL": str(os.cpu_count())})
        wheels = sorted(glob.glob("/content/f2c_wheels/fields2cover-*.whl"))
    sh(f"pip install -q --no-deps {wheels[-1]}")
    print(f"installed in {(time.time()-t0)/60:.1f} min")
import fields2cover as f2c
print("Fields2Cover", getattr(f2c, "__version__", "?"))

In [ ]:
# ---- 2. Google Drive for persistent results ----
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/F2C_benchmark/benchmark_v1_2"
os.makedirs(OUT, exist_ok=True)
os.makedirs("/content/bench", exist_ok=True)
%cd /content/bench

## Benchmark code (written to `/content/bench`)

In [ ]:
%%writefile /content/bench/nb_geom.py
# ---- Track, drive-around detours & section-control simulator (pure numpy/shapely) ----
# This cell does not depend on Fields2Cover: it post-processes any path given as a Track.
import numpy as np
import shapely
from shapely.geometry import LineString, Point
from shapely.ops import unary_union


class Track:
    """An ordered drivable path: list of (LineString, kind) with kind in {'swath', 'turn'}.
    Consecutive pieces of the same kind are merged. `n_rev` counts reversing manoeuvres."""

    def __init__(self, segs, n_rev=0):
        merged = []
        for geom, kind in segs:
            pts = np.asarray(geom.coords if hasattr(geom, "coords") else geom, float)[:, :2]
            if len(pts) < 2 or LineString(pts).length < 1e-6:
                continue
            if merged and merged[-1][1] == kind:
                prev = merged[-1][0]
                if np.allclose(prev[-1], pts[0]):
                    pts = pts[1:]
                merged[-1][0] = np.vstack([prev, pts])
            else:
                merged.append([pts, kind])
        self.segs = [(LineString(p), k) for p, k in merged if len(p) >= 2]
        self.n_rev = n_rev

    def pieces(self, kind=None):
        return [g for g, k in self.segs if kind is None or k == kind]

    def length(self, kind=None):
        return sum(g.length for g in self.pieces(kind))

    @property
    def n_turns(self):
        kinds = [k for _, k in self.segs]
        return sum(1 for i in range(1, len(kinds) - 1)
                   if kinds[i] == "turn" and kinds[i - 1] == "swath" and kinds[i + 1] == "swath")

    @property
    def line(self):
        pts = [np.asarray(self.segs[0][0].coords)]
        for g, _ in self.segs[1:]:
            c = np.asarray(g.coords)
            pts.append(c[1:] if np.allclose(pts[-1][-1], c[0]) else c)
        return LineString(np.vstack(pts))

    @property
    def start(self):
        return self.segs[0][0].coords[0]

    @property
    def end(self):
        return self.segs[-1][0].coords[-1]


# ---------------------------------------------------------------------------
# Drive-around detour: straight -> fillet (rho) -> arc around tree (R) -> fillet -> straight
# ---------------------------------------------------------------------------
def _arc(cx, cy, r, a0, a1, step):
    n = max(2, int(np.ceil(abs(a1 - a0) * r / step)) + 1)
    t = np.linspace(a0, a1, n)
    return np.c_[cx + r * np.cos(t), cy + r * np.sin(t)]


def detour_profile(a, R, rho, step=0.5):
    """Detour in the tree frame: tree at the origin, the swath is the line y = a (|a| < R),
    travelling in +x; the detour bulges towards +y. Returns (points, half_length xf)."""
    h = a + rho
    xf = np.sqrt((R + rho) ** 2 - h ** 2)
    th1 = np.arctan2(-h, xf)                     # fillet 1: CCW from -pi/2 to th1
    while th1 <= -np.pi / 2:
        th1 += 2 * np.pi
    f1 = _arc(-xf, h, rho, -np.pi / 2, th1, step)
    phi1 = np.arctan2(h, -xf)                    # tree arc: CW from phi1 to phi2
    if phi1 < 0:
        phi1 += 2 * np.pi
    phi2 = np.pi - phi1
    while phi2 >= phi1:
        phi2 -= 2 * np.pi
    tc = _arc(0.0, 0.0, R, phi1, phi2, step)
    th2 = np.arctan2(-h, -xf)                    # fillet 2: CCW from th2 to -pi/2
    e2 = -np.pi / 2
    while e2 <= th2:
        e2 += 2 * np.pi
    f2 = _arc(xf, h, rho, th2, e2, step)
    return np.vstack([f1, tc[1:], f2[1:]]), xf


def _enclose(c1, c2):
    """Smallest circle (x, y, R) enclosing two circles."""
    (x1, y1, r1), (x2, y2, r2) = c1, c2
    d = np.hypot(x2 - x1, y2 - y1)
    if d + r2 <= r1:
        return c1
    if d + r1 <= r2:
        return c2
    R = (d + r1 + r2) / 2
    t = (R - r1) / d
    return (x1 + t * (x2 - x1), y1 + t * (y2 - y1), R)


def detour_line(line, circles, rho, is_valid, step=0.5):
    """Splice detours into a straight swath `line` around every clearance circle (x, y, R)
    it crosses. Tries the short side first, then the long side. Returns (LineString, issues)."""
    p0, p1 = np.asarray(line.coords[0], float), np.asarray(line.coords[-1], float)
    L = np.hypot(*(p1 - p0))
    u = (p1 - p0) / L
    n = np.array([-u[1], u[0]])
    to_world = lambda xy: p0 + np.outer(xy[:, 0], u) + np.outer(xy[:, 1], n)

    hits = []
    for c in circles:
        tx, ty = (np.array(c[:2]) - p0) @ u, (np.array(c[:2]) - p0) @ n
        if abs(ty) < c[2] and -c[2] < tx < L + c[2]:
            hits.append(list(c))
    # merge circles whose detours would overlap along the swath
    changed = True
    while changed and len(hits) > 1:
        changed = False
        hits.sort(key=lambda c: (np.array(c[:2]) - p0) @ u)
        for i in range(len(hits) - 1):
            a, b = hits[i], hits[i + 1]
            ta, tb = (np.array(a[:2]) - p0) @ u, (np.array(b[:2]) - p0) @ u
            if tb - ta < a[2] + b[2] + 2 * rho:
                hits[i:i + 2] = [list(_enclose(tuple(a), tuple(b)))]
                changed = True
                break

    pts, issues = [p0], []
    for cx, cy, R in sorted(hits, key=lambda c: (np.array(c[:2]) - p0) @ u):
        tx, ty = (np.array([cx, cy]) - p0) @ u, (np.array([cx, cy]) - p0) @ n
        done = False
        for a in (abs(ty), -abs(ty)):          # short side, then long side
            if abs(a) >= R:
                continue
            s = (-ty / a) if abs(a) > 1e-9 else (1.0 if a >= 0 else -1.0)
            prof, xf = detour_profile(a, R, rho, step)
            if tx - xf < 0 or tx + xf > L:     # detour does not fit inside this swath
                continue
            local = np.c_[tx + prof[:, 0], ty + s * prof[:, 1]]
            piece = to_world(local)
            if is_valid(LineString(piece)):
                pts.extend(piece)
                done = True
                break
        if not done:
            issues.append((cx, cy, R))
    pts.append(p1)
    return LineString(np.asarray(pts)), issues


def make_drive_around(track, trees, margin, width, r_min, allowed_poly, step=0.5):
    """Strategy B3: keep every swath of `track` and drive around trees on the swath itself."""
    circles = [(x, y, max(r + margin + width / 2, r_min)) for x, y, r in trees]
    hard = [(x, y, r + width / 2) for x, y, r in trees]          # canopy + half implement

    def is_valid(piece):
        if not allowed_poly.covers(piece):
            return False
        return all(piece.distance(Point(x, y)) >= R - 1e-6 for x, y, R in hard)

    segs, issues = [], []
    for g, kind in track.segs:
        if kind == "swath":
            c = list(g.coords)
            for a, b in zip(c[:-1], c[1:]):          # each straight piece of the swath run
                g2, iss = detour_line(LineString([a, b]), circles, r_min, is_valid, step)
                segs.append((g2, kind))
                issues += iss
        else:
            segs.append((g, kind))
    return Track(segs, track.n_rev), issues


def max_curvature(line, ds=1.0):
    """Approximate max curvature [1/m] of a polyline (resampled every ds metres)."""
    d = np.arange(0, line.length, ds)
    p = shapely.get_coordinates(shapely.line_interpolate_point(line, d))
    if len(p) < 3:
        return 0.0
    v1, v2 = p[1:-1] - p[:-2], p[2:] - p[1:-1]
    ang = np.abs(np.arctan2(v1[:, 0] * v2[:, 1] - v1[:, 1] * v2[:, 0], (v1 * v2).sum(1)))
    return float(ang.max() / ds)


# ---------------------------------------------------------------------------
# Section-control simulator (raster)
# ---------------------------------------------------------------------------
def simulate_coverage(track, field_shp, obstacles, width, n_sections=6, res=1.0,
                      section_control=True, extra_passes=(), record_window=None):
    """Drive the boom along every working piece of `track` (+ `extra_passes`) and count, per
    raster cell, how many distinct passes applied product to it.
    A section is ON if (section control) the majority of its footprint is inside the field,
    outside every obstacle and not yet covered by an earlier pass; otherwise it is always ON."""
    minx, miny, maxx, maxy = field_shp.buffer(width).bounds
    nx, ny = int(np.ceil((maxx - minx) / res)), int(np.ceil((maxy - miny) / res))
    xs, ys = minx + (np.arange(nx) + 0.5) * res, miny + (np.arange(ny) + 0.5) * res
    X, Y = np.meshgrid(xs, ys)
    inside = shapely.contains_xy(field_shp, X, Y)
    obst = (shapely.contains_xy(obstacles, X, Y)
            if obstacles is not None and not obstacles.is_empty else np.zeros_like(inside))
    target = inside & ~obst
    count = np.zeros((ny, nx), np.int16)
    h = res / 4                     # boom & track sampled at 1/4 cell -> 16 hits per full cell
    full = (res / h) ** 2
    offs = -width / 2 + h / 2 + np.arange(int(round(width / h))) * h
    sec = np.minimum(((offs + width / 2) // (width / n_sections)).astype(int), n_sections - 1)
    sec_n = np.bincount(sec, minlength=n_sections)
    sections_on = sections_total = 0
    states = []                      # (pass id, x, y, dx, dy, on per section) inside record_window

    passes = list(track.pieces("swath")) + list(extra_passes)
    for pid, line in enumerate(passes):
        L = line.length
        if L < res:
            continue
        d = np.arange(0, L, h)
        p = shapely.get_coordinates(shapely.line_interpolate_point(line, d))
        q = shapely.get_coordinates(shapely.line_interpolate_point(line, np.minimum(d + h, L)))
        dv = q - p
        dv[-1] = dv[-2] if len(dv) > 1 else dv[-1]
        dv /= np.maximum(np.hypot(dv[:, 0], dv[:, 1]), 1e-12)[:, None]
        bx = p[:, :1] - dv[:, 1:2] * offs            # (samples, boom points)
        by = p[:, 1:2] + dv[:, 0:1] * offs
        ix, iy = ((bx - minx) / res).astype(int), ((by - miny) / res).astype(int)
        ok = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny)
        ix, iy = np.clip(ix, 0, nx - 1), np.clip(iy, 0, ny - 1)
        if section_control:   # decisions use passes completed before this one
            good = ok & target[iy, ix] & (count[iy, ix] == 0)
            frac = np.stack([good[:, sec == s].mean(1) for s in range(n_sections)], 1)
            on = (frac >= 0.5)[:, sec]
        else:
            on = np.ones_like(ok)
        if record_window is not None:
            m = shapely.contains_xy(record_window, p[:, 0], p[:, 1])
            for i in np.flatnonzero(m):
                states.append((pid, p[i, 0], p[i, 1], dv[i, 0], dv[i, 1], on[i, sec_n.cumsum() - 1].copy()))
        sections_on += on[:, sec_n.cumsum() - 1].sum()
        sections_total += on.shape[0] * n_sections
        flat = (iy * nx + ix)[on & ok]
        cells, hits = np.unique(flat, return_counts=True)
        # applied if > half the cell is under an ON section; an exact half goes to the first pass
        claim = (hits > full / 2) | ((hits == full / 2) & (count.flat[cells] == 0))
        count.flat[cells[claim]] += 1

    cell = res * res
    t = target.sum()
    return dict(
        count=count, target=target, inside=inside, obst=obst,
        extent=(minx, minx + nx * res, miny, miny + ny * res),
        target_ha=t * cell / 1e4,
        missed_pct=100 * (target & (count == 0)).sum() / t,
        overlap_pct=100 * (target & (count >= 2)).sum() / t,
        offtarget_m2=float(((~target) & (count > 0)).sum() * cell),
        applied_ha=float(count.sum() * cell / 1e4),
        sections_on_pct=100 * sections_on / max(sections_total, 1),
        states=states, offs=offs, sec=sec,
    )


def turn_length_inside(track, poly):
    """Non-working distance driven inside `poly` (e.g. over the crop)."""
    return sum(g.intersection(poly).length for g in track.pieces("turn"))


def length_outside(track, poly):
    """Distance driven outside `poly` (e.g. outside the paddock)."""
    return sum(g.difference(poly).length for g, _ in track.segs)


def tree_clearance(track, trees, width):
    """Edge clearance (distance to trunk - canopy radius - half implement) per tree."""
    ln = track.line
    return [ln.distance(Point(x, y)) - r - width / 2 for x, y, r in trees]

In [ ]:
%%writefile /content/bench/bench_geom.py
"""Pure-geometry layer of the benchmark (numpy + shapely only, no Fields2Cover).

- synthetic paddocks (rectangle / convex irregular / L-shape) at a target area
- obstacles: trees (discs) and rock outcrops (polygons), Poisson or Thomas-cluster placement
- general B3 detour around a smooth convex set S (covers discs and buffered polygon hulls)
- polygon-aware clearance
"""
import zlib
import numpy as np
import shapely
from shapely import affinity
from shapely.geometry import LineString, Point, Polygon, box
from shapely.ops import unary_union, nearest_points, substring

from nb_geom import Track, simulate_coverage, turn_length_inside, length_outside, max_curvature  # noqa: F401


def stable_seed(*parts):
    return zlib.crc32("|".join(map(str, parts)).encode()) & 0x7FFFFFFF


# ---------------------------------------------------------------------------
# Paddocks
# ---------------------------------------------------------------------------
def make_field(shape, area_ha, rng):
    """Polygon of `area_ha` hectares with the road along its southern (min-y) side."""
    A = area_ha * 1e4
    if shape == "rectangle":
        poly = box(0, 0, np.sqrt(A * 1.25), np.sqrt(A / 1.25))
    elif shape == "L":
        poly = Polygon([(0, 0), (600, 0), (600, 250), (300, 250), (300, 500), (0, 500)])
    elif shape == "convex":
        n = 7
        ang = np.sort(rng.uniform(0, 2 * np.pi, n))
        rad = rng.uniform(0.8, 1.2, n)
        pts = np.c_[1.4 * rad * np.cos(ang), rad * np.sin(ang)]
        poly = Polygon(pts).convex_hull
    else:
        raise ValueError(shape)
    poly = affinity.scale(poly, np.sqrt(A / poly.area), np.sqrt(A / poly.area), origin=(0, 0))
    minx, miny, _, _ = poly.bounds
    return shapely.normalize(affinity.translate(poly, -minx, -miny))


def road_gate(field):
    """Road-side gate: the boundary point nearest to a road far to the south."""
    minx, miny, maxx, _ = field.bounds
    p = nearest_points(field.exterior, Point((minx + maxx) / 2, miny - 1e4))[0]
    return (p.x, p.y)


def random_vertex_gate(field, rng):
    c = list(field.exterior.coords)[:-1]
    return tuple(c[rng.integers(len(c))])


# ---------------------------------------------------------------------------
# Obstacles
# ---------------------------------------------------------------------------
def _tree(c, rng):
    r = rng.uniform(4, 9)
    return {"type": "tree", "poly": Point(c).buffer(r, quad_segs=16), "r": r}


def _outcrop(c, rng):
    R = rng.uniform(6, 15)
    nv = int(rng.integers(6, 10))
    ang = np.sort(rng.uniform(0, 2 * np.pi, nv))
    rad = R * rng.uniform(0.55, 1.0, nv)
    poly = Polygon(np.c_[c[0] + rad * np.cos(ang), c[1] + rad * np.sin(ang)]).buffer(0)
    return {"type": "outcrop", "poly": poly, "r": R}


def make_obstacles(region, n, otype, pattern, rng, sigma=15.0, mu=4):
    """n obstacles with centres in `region`; pattern 'poisson' (uniform) or 'thomas' (clustered)."""
    if n <= 0 or region.is_empty:
        return []
    minx, miny, maxx, maxy = region.bounds

    def uniform_pt():
        while True:
            p = (rng.uniform(minx, maxx), rng.uniform(miny, maxy))
            if region.contains(Point(p)):
                return p

    if pattern == "poisson":
        centres = [uniform_pt() for _ in range(n)]
    elif pattern == "thomas":
        parents = [uniform_pt() for _ in range(max(1, int(round(n / mu))))]
        centres = []
        while len(centres) < n:
            px, py = parents[rng.integers(len(parents))]
            p = (px + rng.normal(0, sigma), py + rng.normal(0, sigma))
            if region.contains(Point(p)):
                centres.append(p)
    else:
        raise ValueError(pattern)
    out = []
    for c in centres:
        kind = otype if otype in ("tree", "outcrop") else ("tree" if rng.random() < 0.5 else "outcrop")
        out.append(_tree(c, rng) if kind == "tree" else _outcrop(c, rng))
    return out


def obstacle_groups(obstacles, margin):
    """Connected components of overlapping safety zones (obstacle ⊕ B(margin))."""
    zones = [o["poly"].buffer(margin) for o in obstacles]
    parent = list(range(len(zones)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    tree = shapely.STRtree(zones)
    for i, z in enumerate(zones):
        for j in tree.query(z, predicate="intersects"):
            if j > i:
                parent[find(i)] = find(int(j))
    groups = {}
    for i in range(len(zones)):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())


def clearance(track, obstacles, width):
    """Edge clearance δ = dist(path, obstacle polygon) − w/2 for each obstacle (canopy / rock)."""
    ln = track.line
    return [ln.distance(o["poly"]) - width / 2 for o in obstacles]


# ---------------------------------------------------------------------------
# General convex detour
# ---------------------------------------------------------------------------
def _arc(cx, cy, r, a0, a1, step):
    n = max(2, int(np.ceil(abs(a1 - a0) * r / step)) + 1)
    t = np.linspace(a0, a1, n)
    return np.c_[cx + r * np.cos(t), cy + r * np.sin(t)]


def _detour_local(S, L, rho, step):
    """Detour in the swath frame (swath = segment (0,0)-(L,0), bulge towards +y) around the
    smooth convex set S. Returns (points, x1, x2) or None if S does not touch the swath line
    or the detour does not fit between the swath ends."""
    G = S.buffer(rho, quad_segs=32)
    xmin, _, xmax, _ = G.bounds
    hl = LineString([(xmin - 1, rho), (xmax + 1, rho)])
    inter = G.exterior.intersection(hl)
    xs = [p.x for p in getattr(inter, "geoms", [inter]) if not p.is_empty and p.geom_type == "Point"]
    if len(xs) < 2:
        return None
    x1, x2 = min(xs), max(xs)
    if x1 < 0 or x2 > L:
        return None
    c1, c2 = np.array([x1, rho]), np.array([x2, rho])
    T1 = np.array(nearest_points(S, Point(c1))[0].coords[0])
    T2 = np.array(nearest_points(S, Point(c2))[0].coords[0])
    th1 = np.arctan2(*(T1 - c1)[::-1])
    while th1 <= -np.pi / 2:
        th1 += 2 * np.pi
    f1 = _arc(x1, rho, rho, -np.pi / 2, th1, step)
    th2 = np.arctan2(*(T2 - c2)[::-1])
    e2 = -np.pi / 2
    while e2 <= th2:
        e2 += 2 * np.pi
    f2 = _arc(x2, rho, rho, th2, e2, step)
    ring = LineString(S.exterior.coords)
    d1, d2 = ring.project(Point(T1)), ring.project(Point(T2))
    lo, hi = sorted((d1, d2))
    a = substring(ring, lo, hi)
    b_parts = [substring(ring, hi, ring.length), substring(ring, 0, lo)]
    b = LineString(np.vstack([np.asarray(p.coords) for p in b_parts if p.length > 0]))
    top = max((a, b), key=lambda g: np.asarray(g.coords)[:, 1].mean())
    tc = np.asarray(top.coords)
    if np.hypot(*(tc[0] - T1)) > np.hypot(*(tc[-1] - T1)):
        tc = tc[::-1]
    return np.vstack([f1, tc, f2]), x1, x2


def detour_line_convex(line, sets, rho, is_valid, step=0.5, mergeable=None, max_iter=12):
    """Splice detours into straight swath `line` around every smooth convex set in `sets`
    (shapely polygons) that touches it. Returns (LineString, n_unplaced).

    All merging happens BEFORE any detour is committed, so a merged hull can never reach back
    over a detour that was already placed (that produced a 180-degree backtrack in v1.0):
      1. sets whose detour intervals could overlap along the swath are merged into their hull;
      2. every set is tried on both sides; if a set fails, the sets of `mergeable` near it are
         merged into it and the whole list is re-checked from step 1;
      3. a set that still fails is left unplaced (the swath drives straight; it is flagged).
    The committed detours are verified to be strictly ordered along the swath."""
    p0, p1 = np.asarray(line.coords[0], float), np.asarray(line.coords[-1], float)
    L = np.hypot(*(p1 - p0))
    u = (p1 - p0) / L
    n = np.array([-u[1], u[0]])
    ang = np.degrees(np.arctan2(u[1], u[0]))
    to_local = lambda g: affinity.rotate(affinity.translate(g, -p0[0], -p0[1]), -ang, origin=(0, 0))
    to_world = lambda xy: p0 + np.outer(xy[:, 0], u) + np.outer(xy[:, 1], n)
    axis = LineString([(-1e6, 0), (1e6, 0)])
    on_swath = lambda s: s.intersects(axis) and s.bounds[2] > -rho and s.bounds[0] < L + rho

    def merge_overlapping(hs):
        hs = sorted(hs, key=lambda s: s.bounds[0])
        out = []
        for s in hs:
            if out and out[-1].bounds[2] + 2 * rho > s.bounds[0]:
                out[-1] = unary_union([out[-1], s]).convex_hull
            else:
                out.append(s)
        return out if len(out) == len(hs) else merge_overlapping(out)

    def best_side(S):
        best = None
        for flip in (1, -1):
            res = _detour_local(affinity.scale(S, 1, flip, origin=(0, 0)), L, rho, step)
            if res is None:
                continue
            prof = res[0] * np.array([1, flip])
            piece = LineString(to_world(prof))
            if is_valid(piece) and (best is None or piece.length < best[0]):
                best = (piece.length, prof, res[1], res[2])
        return best

    loc_merge = [to_local(g) for g in (mergeable or [])]
    hits = [s for s in (to_local(s) for s in sets) if on_swath(s)]
    unplaced_sets = []
    for _ in range(max_iter):
        hits = merge_overlapping(hits)
        placed, failed = [], None
        for S in hits:
            b = best_side(S)
            if b is None:
                failed = S
                break
            placed.append(b)
        if failed is None:
            break
        extra = [m for m in loc_merge if m.intersects(failed.buffer(2 * rho)) and not m.within(failed)]
        grown = unary_union([failed] + extra).convex_hull if extra else None
        hits = [s for s in hits if s is not failed]
        if grown is not None and grown.area > failed.area * 1.001:
            hits.append(grown)            # retry with the grown hull (re-merged in step 1)
        else:
            unplaced_sets.append(failed)  # give up on this one: straight through, flagged
    else:
        placed = [b for b in (best_side(S) for S in merge_overlapping(hits)) if b is not None]

    placed.sort(key=lambda b: b[2])
    pts, x_prev = [np.zeros(2)], 0.0
    for _, prof, x1, x2 in placed:
        if x1 < x_prev - 1e-6:            # should be impossible after merging; never backtrack
            unplaced_sets.append(None)
            continue
        pts.extend(prof)
        x_prev = x2
    pts.append(np.array([L, 0.0]))
    P = np.asarray(pts)
    keep = np.r_[True, np.hypot(*np.diff(P, axis=0).T) > 1e-6]   # drop duplicate vertices at joins
    return LineString(to_world(P[keep])), len(unplaced_sets)


def detour_sets(obstacles, margin, width, r_min, quad_segs=16):
    """Smooth convex clearance set per obstacle group:
    discs (single tree): disc of radius max(r + m + w/2, r_min);
    otherwise: convex hull of the group ⊕ B(max(m + w/2, r_min))."""
    eps = margin + width / 2
    sets = []
    for g in obstacle_groups(obstacles, margin):
        if len(g) == 1 and obstacles[g[0]]["type"] == "tree":
            o = obstacles[g[0]]
            c = o["poly"].centroid
            sets.append(c.buffer(max(o["r"] + eps, r_min), quad_segs=quad_segs * 2))
        else:
            hull = unary_union([obstacles[i]["poly"] for i in g]).convex_hull
            sets.append(hull.buffer(max(eps, r_min), quad_segs=quad_segs))
    return sets


def make_drive_around_poly(track, obstacles, margin, width, r_min, allowed_poly, step=0.5):
    """Strategy B3 for discs and polygons. Returns (Track, n_unplaced_detours)."""
    sets = detour_sets(obstacles, margin, width, r_min)
    hard = [o["poly"].buffer(width / 2 - 1e-6) for o in obstacles]
    hard_tree = shapely.STRtree(hard) if hard else None

    def is_valid(piece):
        if not allowed_poly.covers(piece):
            return False
        return hard_tree is None or len(hard_tree.query(piece, predicate="intersects")) == 0

    segs, unplaced = [], 0
    for g, kind in track.segs:
        if kind != "swath":
            segs.append((g, kind))
            continue
        c = list(g.coords)
        for a, b in zip(c[:-1], c[1:]):
            g2, u = detour_line_convex(LineString([a, b]), sets, r_min, is_valid, step, mergeable=sets)
            segs.append((g2, kind))
            unplaced += u
    return Track(segs, track.n_rev), unplaced


def corridor_passes(region, width, min_area=None):
    """Working passes that cover strip-shaped `region` pieces (e.g. the corridors F2C's
    HG_Corridor_gen cuts between decomposed cells, which no swath covers). Each piece gets
    ceil(strip width / w) passes along its long axis (minimum rotated rectangle)."""
    out = []
    min_area = width * width if min_area is None else min_area
    for piece in getattr(region, "geoms", [region]):
        if piece.is_empty or piece.area < min_area:
            continue
        rect = np.asarray(piece.minimum_rotated_rectangle.exterior.coords)[:4]
        e1, e2 = rect[1] - rect[0], rect[2] - rect[1]
        long_e = e1 if np.hypot(*e1) >= np.hypot(*e2) else e2
        u = long_e / np.hypot(*long_e)
        nrm = np.array([-u[1], u[0]])
        xy = np.asarray(piece.exterior.coords)
        a, b = (xy @ nrm).min(), (xy @ nrm).max()
        k = max(1, int(np.ceil((b - a) / width - 1e-6)))
        for j in range(k):
            c = (a + b) / 2 + (j - (k - 1) / 2) * width
            ln = LineString([c * nrm - 1e5 * u, c * nrm + 1e5 * u]).intersection(piece.buffer(0.01))
            for seg in getattr(ln, "geoms", [ln]):
                if seg.geom_type == "LineString" and seg.length > width:
                    out.append(seg)
    return out


def clean_poly(g, grid=1e-6):
    """Snap to a fine grid and repair: F2C/OGR output can carry repeated or near-coincident vertices
    (zero-length segments) that make GEOS overlay ops raise 'non-noded intersection'."""
    g = shapely.make_valid(shapely.set_precision(g, grid))
    polys_ = [p for p in getattr(g, "geoms", [g]) if p.geom_type in ("Polygon", "MultiPolygon")]
    return unary_union(polys_) if polys_ else Polygon()

In [ ]:
%%writefile /content/bench/f2c_bench.py
"""Headless benchmark runner for obstacle-aware coverage planning (R2-R5).

Runs in Colab (Fields2Cover needs Linux). Typical use from the driver notebook:
    python f2c_bench.py design  --out RESULTS            # writes RESULTS/design.csv + run_meta.json
    python f2c_bench.py run     --out RESULTS --workers 2  # plans every pending instance (resumable)
    python f2c_bench.py merge   --out RESULTS            # RESULTS/runs.csv
Each instance is planned in its own subprocess, so a C++ crash only loses that instance
(its error is written to RESULTS/parts/<id>.err).
"""
import argparse
import csv
import json
import math
import os
import platform
import subprocess
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
BENCH_VERSION = "benchmark_v1 / f2c_bench 1.4"

CONFIG = dict(
    area_ha=20.0,
    shapes=["rectangle", "convex", "L"],
    densities=[0.0, 0.5, 1.0, 2.0],             # obstacles per ha
    obstacle_types=["tree", "outcrop", "mixed"],
    patterns=["poisson", "thomas"],
    widths={12: 6, 24: 12, 36: 18},             # implement width -> number of sections
    seeds=[0, 1, 2, 3, 4],
    gate_subset=dict(shape="rectangle", densities=[0.5, 2.0], width=24, otype="tree", pattern="poisson"),
    r_min=6.0, max_diff_curv=0.1, cruise_vel=2.5, turn_vel=1.2, reverse_penalty_s=10.0,
    margin=3.0, k_min=2, rp_time=10, rp_optimum=True, allow_reverse=True,
    turn_discretization=0.2,
    sg_allow_overlap=False,    # F2C flag; has no effect on generateBestSwaths in v2.1.1 (verified)
    fill_remainders=True,      # add an overlapping swath where a strip < w is left beside the last swath   # m between turn points (F2C default 0.01 -> ~20x fewer path points)
    raster_res=1.0, cover_headlands=True, placement_inset=10.0,
    # SC-aware selection criterion C = c_t*T[min] + c_p*applied_SC[ha] + c_m*missed_area[ha]  (R4)
    c_t=2.5,       # cost per minute of machine time
    c_p=40.0,      # cost per hectare of product applied
    c_m=300.0,     # cost per hectare left untreated
    angle_by_C=False,   # also pick the swath angle by C (A and B3), from {NSwath, SwathLength, NSwath+90°}
    geojson_seeds=[0],  # dump geometry for instances with these seeds
    instance_timeout_s=1800,
)

COLUMNS = ("instance_id seed shape area_ha density_per_ha n_obstacles obstacle_type pattern width_m cov_width_m "
           "n_sections r_min_m gate_type strategy sc feasible n_unplaced_detours decomp_failed swath_angle_deg "
           "n_swaths turns reversals L_work_m L_turn_m L_total_m eta_pct time_min over_crop_m outside_paddock_m "
           "min_clearance_m max_curvature missed_pct overlap_pct offtarget_m2 applied_ha product_saved_pct "
           "sections_on_pct t_hg_s t_sg_s t_rp_s t_pp_s t_detour_s t_sim_s "
           "crit_C selected turn_model k_headland gate_x gate_y n_groups min_clearance_work_m min_clearance_turn_m error").split()


# ---------------------------------------------------------------------------
# Design
# ---------------------------------------------------------------------------
def make_design(cfg):
    rows = []

    def add(shape, d, otype, pat, w, gate, seed):
        iid = f"{shape}_{otype}_{pat}_d{d:g}_w{w}_{gate}_s{seed}"
        rows.append(dict(instance_id=iid, shape=shape, density=d, otype=otype, pattern=pat, width=w,
                         n_sections=cfg["widths"][w], gate_type=gate, seed=seed))

    for shape in cfg["shapes"]:
        for w in cfg["widths"]:
            for seed in cfg["seeds"]:
                for d in cfg["densities"]:
                    if d == 0:
                        add(shape, 0.0, "none", "none", w, "road", seed)
                        continue
                    for otype in cfg["obstacle_types"]:
                        for pat in cfg["patterns"]:
                            add(shape, d, otype, pat, w, "road", seed)
    gs = cfg["gate_subset"]
    for d in gs["densities"]:
        for seed in cfg["seeds"]:
            add(gs["shape"], d, gs["otype"], gs["pattern"], gs["width"], "random", seed)
    return rows


def run_meta(cfg):
    meta = {"bench_version": BENCH_VERSION, "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "python": platform.python_version(), "config": cfg}
    try:
        import fields2cover as f2c
        meta["fields2cover"] = getattr(f2c, "__version__", "unknown")
    except Exception:
        meta["fields2cover"] = "not importable here"
    try:
        import shapely
        meta["shapely"] = shapely.__version__
    except Exception:
        pass
    try:
        meta["git"] = subprocess.run(["git", "-C", HERE, "rev-parse", "HEAD"], capture_output=True,
                                     text=True).stdout.strip() or "not a git repo"
    except Exception:
        meta["git"] = "not a git repo"
    import hashlib
    meta["source_sha256"] = {f: hashlib.sha256(open(os.path.join(HERE, f), "rb").read()).hexdigest()[:16]
                             for f in ("f2c_bench.py", "bench_geom.py", "nb_geom.py")}
    return meta


# ---------------------------------------------------------------------------
# Fields2Cover helpers (copied from the tutorial; see SWIG memory note there)
# ---------------------------------------------------------------------------
def _f2c():
    import fields2cover as f2c
    return f2c


def to_shapely(g):
    from shapely import wkt, force_2d
    return force_2d(wkt.loads(g.exportToWkt()))


def make_ring(f2c, coords):
    coords = [(float(x), float(y)) for x, y in coords]
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    return f2c.LinearRing(f2c.VectorPoint([f2c.Point(x, y) for x, y in coords]))


def polys(g):
    return list(g.geoms) if hasattr(g, "geoms") else [g]


def track_from_path(f2c, path):
    from bench_geom import Track
    pts, kinds, rev = [], [], []
    for i in range(path.size()):
        s = path.getState(i)
        pts.append((s.point.getX(), s.point.getY()))
        kinds.append("swath" if s.type == f2c.PathSectionType_SWATH else "turn")
        rev.append(s.dir == f2c.PathDirection_BACKWARD)
    e = path.atEnd()
    pts.append((e.getX(), e.getY()))
    segs = [([pts[i], pts[i + 1]], kinds[i]) for i in range(len(kinds)) if pts[i] != pts[i + 1]]
    n_rev = sum(1 for i, r in enumerate(rev) if r and (i == 0 or not rev[i - 1]))
    return Track(segs, n_rev)


class NoSwaths(RuntimeError):
    """Nothing to route (empty mainland / headland track / zero swaths). F2C 2.1.1's genRoute does not
    throw on this: OR-Tools CHECK-aborts the whole process (depot id -1), so we must guard first."""


class Planner:
    """All F2C stages for one instance, with per-stage timers."""

    def __init__(self, cfg, field_shp, width, gate):
        self.f2c = f2c = _f2c()
        self.cfg, self.field_shp, self.w, self.gate = cfg, field_shp, width, gate
        r = cfg["r_min"]
        d_omega = r + math.sqrt(max(4 * r * r - (r + width / 2) ** 2, 0)) if width < 2 * r else r
        self.k = max(cfg["k_min"], math.ceil((d_omega + width / 2) / width))
        rb = f2c.Robot(width, width)
        rb.setMinTurningRadius(r)
        rb.setMaxDiffCurv(cfg["max_diff_curv"])
        rb.setCruiseVel(cfg["cruise_vel"])
        rb.setTurnVel(cfg["turn_vel"])
        self.robot = rb
        self.hl = f2c.HG_Const_gen()
        self.bf = f2c.SG_BruteForce()
        self.bf.setAllowOverlap(bool(cfg.get("sg_allow_overlap", False)))
        self.pp = f2c.PP_PathPlanning()

    def cells(self, holes=()):
        f2c = self.f2c
        cells = f2c.Cells(f2c.Cell(make_ring(f2c, list(self.field_shp.exterior.coords))))
        for h in holes:
            cells.addRing(0, make_ring(f2c, list(h.exterior.coords)))
        return cells

    def headlands(self, cells):
        t0 = time.perf_counter()
        mid = self.hl.generateHeadlands(cells, 0.5 * self.k * self.w)
        no = self.hl.generateHeadlands(cells, self.k * self.w)
        return mid, no, time.perf_counter() - t0

    def headland_passes(self, cells, n_rounds=None):
        from shapely.geometry import LineString
        if not self.cfg["cover_headlands"]:
            return []
        n = int(self.k) if n_rounds is None else int(n_rounds)
        # 1.4: offset rings in shapely. F2C 2.1.1's generateHeadlandSwaths returns malformed rings
        # (self-crossing, even outside the field) once the bands of many holes merge, which left
        # B1's obstacle bands unworked (B1 missed% ~18-50 % at w=36, d=2 in benchmark_v1_1).
        from bench_geom import clean_poly
        region = clean_poly(to_shapely(cells))
        out = []
        for i in range(max(1, n)):
            band = region.buffer(-(i + 0.5) * self.w)
            for p in polys(band):
                if p.is_empty or p.geom_type != "Polygon":
                    continue
                out += [LineString(p.exterior.coords)] + [LineString(r.coords) for r in p.interiors]
        return out

    def swaths(self, no_hl, angle=None):
        t0 = time.perf_counter()
        obj = self.f2c.OBJ_NSwathModified()
        if no_hl.size() == 0 or no_hl.area() < 1.0:
            raise NoSwaths("no swaths: empty mainland")
        big = max(range(no_hl.size()), key=lambda i: no_hl.getGeometry(i).area())
        if angle is None:
            angle = self.bf.computeBestAngle(obj, self.w, no_hl.getGeometry(big))
            sw = self.bf.generateBestSwaths(obj, self.w, no_hl)
        else:
            sw = self.bf.generateSwaths(angle, self.w, no_hl)
        if self.cfg.get("fill_remainders", True):
            self.n_fill = self.fill_remainders(no_hl, sw)
        return sw, angle, time.perf_counter() - t0

    def fill_remainders(self, no_hl, sw, tol=0.5):
        """F2C's brute-force generator spaces swaths w apart from one side of a cell, which leaves an
        uncovered strip (< w) along the far side. Add one overlapping swath along each such strip.
        Mutates `sw` in place (sw is alive, so sw.at(ci) is a safe reference)."""
        from shapely.geometry import LineString
        from shapely.ops import unary_union
        f2c, w = self.f2c, self.w
        cells = [to_shapely(no_hl.getGeometry(i)) for i in range(no_hl.size())]
        added = 0
        for ci in range(sw.size()):
            cur = sw.at(ci)
            if cur.size() == 0:
                continue
            lines = [to_shapely(cur.at(i).clone().getPath()) for i in range(cur.size())]
            c0 = np.asarray(lines[0].coords)
            u = (c0[-1] - c0[0]) / np.linalg.norm(c0[-1] - c0[0])
            nrm = np.array([-u[1], u[0]])
            mid = lines[0].interpolate(0.5, normalized=True)
            cell = min(cells, key=lambda c: c.distance(mid))
            covered = unary_union([l.buffer(w / 2, cap_style="flat") for l in lines])
            unc = cell.difference(covered).buffer(-tol / 2).buffer(tol / 2)
            for piece in polys(unc):
                if piece.is_empty or piece.area < w * w:
                    continue
                xy = np.asarray(piece.exterior.coords)
                if np.ptp(xy @ u) < 2 * w:          # small corner wedges are left to the headland
                    continue
                cpos = 0.5 * ((xy @ nrm).min() + (xy @ nrm).max())
                base = cpos * nrm
                cut = LineString([base - 1e5 * u, base + 1e5 * u]).intersection(cell)
                for seg in getattr(cut, "geoms", [cut]):
                    if seg.geom_type != "LineString" or seg.length < w or not seg.intersects(piece.buffer(0.1)):
                        continue
                    ls = f2c.LineString()
                    for x, y in seg.coords:
                        ls.addPoint(float(x), float(y))
                    cur.emplace_back(ls, w, 100000 + added)
                    added += 1
        return added

    def route(self, mid_hl, swaths):
        from shapely.geometry import Point
        f2c = self.f2c
        t0 = time.perf_counter()
        if swaths.sizeTotal() == 0:
            raise NoSwaths("no swaths")
        if mid_hl.size() == 0 or mid_hl.area() < 1.0:
            raise NoSwaths("no swaths: empty headland track")
        poly = max(polys(to_shapely(mid_hl)), key=lambda p: p.area)
        q = poly.exterior.interpolate(poly.exterior.project(Point(self.gate)))
        rp = f2c.RP_RoutePlannerBase()
        rp.setStartAndEndPoint(f2c.Point(q.x, q.y))
        route = rp.genRoute(mid_hl, swaths, False, 1e-4, True, int(self.cfg["rp_time"]), bool(self.cfg["rp_optimum"]))
        return route, time.perf_counter() - t0

    def path(self, route):
        from bench_geom import length_outside
        f2c = self.f2c
        t0 = time.perf_counter()
        models = {"Dubins CC": f2c.PP_DubinsCurvesCC()}
        if self.cfg["allow_reverse"] and self.w < 2 * self.cfg["r_min"]:
            models["Reeds-Shepp HC"] = f2c.PP_ReedsSheppCurvesHC()
        best = None
        for name, tm in models.items():
            tm.setDiscretization(float(self.cfg.get("turn_discretization", 0.01)))
            tr = track_from_path(f2c, self.pp.planPath(self.robot, route, tm))
            score = (tr.length("turn") / self.cfg["turn_vel"] + self.cfg["reverse_penalty_s"] * tr.n_rev
                     + 6.0 * length_outside(tr, self.field_shp))
            if best is None or score < best[0]:
                best = (score, name, tr)
        return best[2], best[1], time.perf_counter() - t0


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def _part_clear(track, obstacles, width, kind):
    """Min edge clearance over the working ('swath') or non-working ('turn') pieces only."""
    pieces = track.pieces(kind)
    if not obstacles or not pieces:
        return ""
    return round(min(g.distance(o["poly"]) for g in pieces for o in obstacles) - width / 2, 3)


def evaluate(cfg, track, *, field_shp, zones, crop, obstacles, width, n_sections, extra):
    """Path metrics (shared by both SC rows) + one simulation per SC setting."""
    from bench_geom import turn_length_inside, length_outside, clearance, max_curvature, simulate_coverage
    L_sw, L_tn = track.length("swath"), track.length("turn")
    L_hl = sum(g.length for g in extra)
    T = (L_sw + L_hl) / cfg["cruise_vel"] + L_tn / cfg["turn_vel"] + cfg["reverse_penalty_s"] * track.n_rev
    base = dict(turns=track.n_turns, reversals=track.n_rev, L_work_m=round(L_sw + L_hl, 1), L_turn_m=round(L_tn, 1),
                L_total_m=round(L_sw + L_hl + L_tn, 1), eta_pct=round(100 * (L_sw + L_hl) / (L_sw + L_hl + L_tn), 2),
                time_min=round(T / 60, 2), over_crop_m=round(turn_length_inside(track, crop.buffer(-1.0)), 1),
                outside_paddock_m=round(length_outside(track, field_shp), 1),
                min_clearance_m=round(min(clearance(track, obstacles, width)), 3) if obstacles else "",
                min_clearance_work_m=_part_clear(track, obstacles, width, "swath"),
                min_clearance_turn_m=_part_clear(track, obstacles, width, "turn"),
                max_curvature=round(max((max_curvature(g) for g in track.pieces("swath")), default=0.0), 4))
    sims = {}
    for sc in (False, True):
        t0 = time.perf_counter()
        r = simulate_coverage(track, field_shp, zones, width, n_sections, cfg["raster_res"], section_control=sc,
                              extra_passes=extra)
        sims[sc] = dict(missed_pct=round(r["missed_pct"], 3), overlap_pct=round(r["overlap_pct"], 3),
                        offtarget_m2=round(r["offtarget_m2"], 1), applied_ha=round(r["applied_ha"], 4),
                        sections_on_pct=round(r["sections_on_pct"], 2), t_sim_s=round(time.perf_counter() - t0, 3),
                        _target_ha=r["target_ha"], _count=r["count"], _extent=r["extent"])
    saved = 100 * (1 - sims[True]["applied_ha"] / sims[False]["applied_ha"]) if sims[False]["applied_ha"] else 0.0
    for sc in sims:
        s = sims[sc]
        missed_ha = s["missed_pct"] / 100 * s["_target_ha"]
        s["crit_C"] = round(cfg["c_t"] * base["time_min"] + cfg["c_p"] * s["applied_ha"] + cfg["c_m"] * missed_ha, 2)
        s["product_saved_pct"] = round(saved, 3)
    return base, sims


# ---------------------------------------------------------------------------
# One instance
# ---------------------------------------------------------------------------
def build_instance(cfg, spec):
    from bench_geom import (make_field, make_obstacles, road_gate, random_vertex_gate, stable_seed)
    field = make_field(spec["shape"], cfg["area_ha"], np.random.default_rng(stable_seed(spec["shape"], spec["seed"])))
    wmax = max(cfg["widths"])
    region = field.buffer(-(cfg["k_min"] * wmax + cfg["placement_inset"]))  # same layout for every width
    n = int(round(spec["density"] * cfg["area_ha"]))
    obs = make_obstacles(region, n, spec["otype"], spec["pattern"],
                         np.random.default_rng(stable_seed(spec["shape"], spec["density"], spec["otype"],
                                                           spec["pattern"], spec["seed"])))
    gate = road_gate(field) if spec["gate_type"] == "road" else \
        random_vertex_gate(field, np.random.default_rng(stable_seed("gate", spec["shape"], spec["seed"])))
    return field, obs, gate


def run_instance(cfg, spec, out_dir):
    from shapely.geometry import mapping, Point
    from shapely.ops import unary_union
    from bench_geom import make_drive_around_poly, obstacle_groups

    field, obs, gate = build_instance(cfg, spec)
    w, ns = int(spec["width"]), int(spec["n_sections"])
    P = Planner(cfg, field, w, gate)
    zones_list = [o["poly"].buffer(cfg["margin"]) for o in obs]
    zones = unary_union(zones_list) if obs else None
    holes = [p.simplify(0.2) for p in polys(zones)] if obs else []
    common = dict(instance_id=spec["instance_id"], seed=spec["seed"], shape=spec["shape"],
                  area_ha=round(field.area / 1e4, 3), density_per_ha=spec["density"], n_obstacles=len(obs),
                  obstacle_type=spec["otype"], pattern=spec["pattern"], width_m=w, cov_width_m=w, n_sections=ns,
                  r_min_m=cfg["r_min"], gate_type=spec["gate_type"], k_headland=P.k,
                  gate_x=round(gate[0], 2), gate_y=round(gate[1], 2),
                  n_groups=len(obstacle_groups(obs, cfg["margin"])) if obs else 0)
    rows, geo = [], {}

    def emit(strategy, track, crop, extra, timings, angle, n_sw, turn_model, unplaced="", decomp_failed="", error=""):
        if track is None:
            for sc in (0, 1):
                rows.append({**common, "strategy": strategy, "sc": sc, "feasible": False,
                             "decomp_failed": decomp_failed, "error": error, **timings})
            return
        base, sims = evaluate(cfg, track, field_shp=field, zones=zones, crop=crop, obstacles=obs, width=w,
                              n_sections=ns, extra=extra)
        ok_clear = base["min_clearance_m"] == "" or base["min_clearance_m"] >= 0
        feasible = bool(ok_clear and not unplaced)
        for sc in (False, True):
            s = sims[sc]
            rows.append({**common, "strategy": strategy, "sc": int(sc), "feasible": feasible,
                         "n_unplaced_detours": unplaced, "decomp_failed": decomp_failed,
                         "swath_angle_deg": round(math.degrees(angle) % 180, 2) if angle is not None else "",
                         "n_swaths": n_sw, **base, **{k: v for k, v in s.items() if not k.startswith("_")},
                         **timings, "turn_model": turn_model, "error": error})
        geo[strategy] = dict(track=track, extra=extra, sims=sims)

    # ---- A (open-paddock plan; reference, infeasible if it drives through obstacles) ----
    cells_A = P.cells()
    mid_A, no_A, t_hg = P.headlands(cells_A)
    HL_A = P.headland_passes(cells_A)
    no_A_shp = to_shapely(no_A)
    cand_angles = [None]
    if cfg["angle_by_C"]:
        big = max(range(no_A.size()), key=lambda i: no_A.getGeometry(i).area())
        a1 = P.bf.computeBestAngle(P.f2c.OBJ_NSwathModified(), w, no_A.getGeometry(big))
        a2 = P.bf.computeBestAngle(P.f2c.OBJ_SwathLength(), w, no_A.getGeometry(big))
        cand_angles = [a1, a2, (a1 + math.pi / 2) % math.pi]
    best_A = None
    for ang in cand_angles:
        sw, angle, t_sg = P.swaths(no_A, ang)
        route, t_rp = P.route(mid_A, sw)
        track, model, t_pp = P.path(route)
        base, sims = evaluate(cfg, track, field_shp=field, zones=zones, crop=no_A_shp, obstacles=obs,
                              width=w, n_sections=ns, extra=HL_A) if len(cand_angles) > 1 else (None, {True: {"crit_C": 0}})
        if best_A is None or sims[True]["crit_C"] < best_A[0]:
            best_A = (sims[True]["crit_C"], track, model, angle, sw.sizeTotal(), t_sg, t_rp, t_pp)
    _, track_A, model_A, angle_A, nsw_A, t_sg, t_rp, t_pp = best_A
    tim_A = dict(t_hg_s=round(t_hg, 3), t_sg_s=round(t_sg, 3), t_rp_s=round(t_rp, 3), t_pp_s=round(t_pp, 3), t_detour_s=0.0)
    crop_A = no_A_shp.difference(zones) if obs else no_A_shp
    emit("A", track_A, crop_A, HL_A, tim_A, angle_A, nsw_A, model_A)

    if obs:
        # ---- B1: obstacles as holes with headland rings ----
        mid_B = cells_B = no_B_shp = None
        try:
            cells_B = P.cells(holes)
            mid_B, no_B, t_hgB = P.headlands(cells_B)
            HL_B = P.headland_passes(cells_B)
            no_B_shp = to_shapely(no_B)
            sw, angle, t_sgB = P.swaths(no_B)
            route, t_rpB = P.route(mid_B, sw)
            track, model, t_ppB = P.path(route)
            emit("B1", track, no_B_shp, HL_B, dict(t_hg_s=round(t_hgB, 3), t_sg_s=round(t_sgB, 3),
                 t_rp_s=round(t_rpB, 3), t_pp_s=round(t_ppB, 3), t_detour_s=0.0), angle, sw.sizeTotal(), model)
        except Exception as e:
            emit("B1", None, None, None, {}, None, "", "", error=str(e) if isinstance(e, NoSwaths) else f"{type(e).__name__}: {e}")

        # ---- B2: Boustrophedon decomposition ----
        stage = "headlands"
        try:
            if mid_B is None or no_B_shp is None:
                raise RuntimeError("B1 headlands unavailable")
            if mid_B.size() == 0 or mid_B.area() < 1.0:
                raise NoSwaths("no swaths: empty headland track")
            stage = "decomp"
            t0 = time.perf_counter()
            dec = P.f2c.DECOMP_Boustrophedon()
            dec.setSplitAngle(angle_A)
            dcells = dec.decompose(mid_B)
            no_B2 = P.f2c.HG_Corridor_gen().generateHeadlands(dcells, 0.5 * P.k * w)
            # 1.4: HG_Corridor_gen only opens corridors along SHARED cell borders, so outer and hole
            # edges stayed on the mid-headland track: swaths ended where the turns start, and at
            # k*w <= 2*r_min (w=12) the Dubins loops left the field / entered obstacle buffers.
            # Clip to the B1 mainland so every outer/hole edge keeps the full k*w turning band.
            no_B2 = no_B2.intersection(no_B)
            t_hg2 = time.perf_counter() - t0
            stage = "plan"
            sw, angle, t_sg2 = P.swaths(no_B2)
            route, t_rp2 = P.route(mid_B, sw)
            track, model, t_pp2 = P.path(route)
            HL_B2 = P.headland_passes(cells_B)
            # corridors between decomposed cells are travel lanes in F2C's recipe; work them as passes
            from bench_geom import corridor_passes, clean_poly
            gaps = clean_poly(no_B_shp).difference(clean_poly(to_shapely(no_B2)))
            gaps = gaps.buffer(-0.25).buffer(0.25)
            HL_B2 = HL_B2 + corridor_passes(gaps, w)
            emit("B2", track, no_B_shp, HL_B2, dict(t_hg_s=round(t_hg2, 3), t_sg_s=round(t_sg2, 3),
                 t_rp_s=round(t_rp2, 3), t_pp_s=round(t_pp2, 3), t_detour_s=0.0), angle, sw.sizeTotal(), model,
                 decomp_failed=False)
        except Exception as e:
            emit("B2", None, None, None, {}, None, "", "", decomp_failed=(stage == "decomp" and not isinstance(e, NoSwaths)),
                 error=str(e) if isinstance(e, NoSwaths) else f"{type(e).__name__}: {e}")

        # ---- B3: A's plan + detours (t_hg..t_pp are A's, since B3 reuses A's plan) ----
        t0 = time.perf_counter()
        track_B3, unplaced = make_drive_around_poly(track_A, obs, cfg["margin"], w, cfg["r_min"], field.buffer(-w / 2))
        emit("B3", track_B3, crop_A, HL_A, {**tim_A, "t_detour_s": round(time.perf_counter() - t0, 3)},
             angle_A, nsw_A, model_A, unplaced=unplaced)

    # ---- selection by C among feasible obstacle strategies (SC on) ----
    cands = [r for r in rows if r["sc"] == 1 and r.get("feasible") and (r["strategy"] != "A" or not obs)]
    best = min(cands, key=lambda r: r["crit_C"])["strategy"] if cands else ""
    for r in rows:
        r["selected"] = int(r["strategy"] == best)

    # ---- outputs ----
    os.makedirs(os.path.join(out_dir, "parts"), exist_ok=True)
    part = os.path.join(out_dir, "parts", spec["instance_id"] + ".csv")
    with open(part + ".tmp", "w", newline="") as fh:
        wr = csv.DictWriter(fh, fieldnames=COLUMNS, extrasaction="ignore")
        wr.writeheader()
        wr.writerows(rows)
    os.replace(part + ".tmp", part)
    if os.path.exists(part[:-4] + ".err"):
        os.remove(part[:-4] + ".err")
    if int(spec["seed"]) in cfg["geojson_seeds"]:
        write_geojson(os.path.join(out_dir, "geojson", spec["instance_id"]), field, obs, zones, gate, geo, w)
    return rows


def write_geojson(stem, field, obs, zones, gate, geo, width):
    """Geometry dump + coverage-count rasters (npz) for figures (R3, R6)."""
    from shapely.geometry import mapping, Point
    os.makedirs(os.path.dirname(stem), exist_ok=True)
    feats = [{"type": "Feature", "properties": {"layer": "paddock"}, "geometry": mapping(field)},
             {"type": "Feature", "properties": {"layer": "gate"}, "geometry": mapping(Point(gate))}]
    feats += [{"type": "Feature", "properties": {"layer": "obstacle", "kind": o["type"], "id": i},
               "geometry": mapping(o["poly"])} for i, o in enumerate(obs)]
    if zones is not None:
        feats.append({"type": "Feature", "properties": {"layer": "zone"}, "geometry": mapping(zones)})
    arrays = {}
    for strat, g in geo.items():
        feats += [{"type": "Feature", "properties": {"layer": "path", "strategy": strat, "kind": k, "seq": j},
                   "geometry": mapping(ln)} for j, (ln, k) in enumerate(g["track"].segs)]
        feats += [{"type": "Feature", "properties": {"layer": "headland_pass", "strategy": strat},
                   "geometry": mapping(ln)} for ln in g["extra"]]
        for sc in (False, True):
            arrays[f"{strat}_sc{int(sc)}"] = g["sims"][sc]["_count"]
            arrays["extent"] = np.array(g["sims"][sc]["_extent"])
    with open(stem + ".geojson", "w") as fh:
        json.dump({"type": "FeatureCollection", "properties": {"width": width}, "features": feats}, fh)
    np.savez_compressed(stem + "_coverage.npz", **arrays)


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def load_cfg(out):
    p = os.path.join(out, "run_meta.json")
    if os.path.exists(p):
        cfg = json.load(open(p))["config"]
        cfg["widths"] = {int(k): v for k, v in cfg["widths"].items()}
        return cfg
    return CONFIG


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["design", "run", "one", "merge"])
    ap.add_argument("--out", required=True)
    ap.add_argument("--id")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--limit", type=int, default=0, help="plan at most N pending instances")
    ap.add_argument("--filter", default="", help="only instance ids containing this text")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    if a.cmd == "design":
        rows = make_design(CONFIG)
        with open(os.path.join(a.out, "design.csv"), "w", newline="") as fh:
            wr = csv.DictWriter(fh, fieldnames=list(rows[0]))
            wr.writeheader(); wr.writerows(rows)
        json.dump(run_meta(CONFIG), open(os.path.join(a.out, "run_meta.json"), "w"), indent=1)
        print(f"{len(rows)} instances -> {a.out}/design.csv")

    elif a.cmd == "one":
        cfg = load_cfg(a.out)
        spec = next(r for r in csv.DictReader(open(os.path.join(a.out, "design.csv"))) if r["instance_id"] == a.id)
        spec["density"] = float(spec["density"]); spec["seed"] = int(spec["seed"])
        try:
            run_instance(cfg, spec, a.out)
        except Exception:
            os.makedirs(os.path.join(a.out, "parts"), exist_ok=True)
            open(os.path.join(a.out, "parts", a.id + ".err"), "w").write(traceback.format_exc())
            raise

    elif a.cmd == "run":
        cfg = load_cfg(a.out)
        design = list(csv.DictReader(open(os.path.join(a.out, "design.csv"))))
        done = lambda i: os.path.exists(os.path.join(a.out, "parts", i + ".csv"))
        todo = [r["instance_id"] for r in design if a.filter in r["instance_id"] and not done(r["instance_id"])]
        if a.limit:
            todo = todo[:a.limit]
        print(f"{len(todo)} pending instances, {a.workers} workers", flush=True)
        t_start = time.time()

        def job(iid):
            t0 = time.time()
            try:
                p = subprocess.run([sys.executable, os.path.abspath(__file__), "one", "--out", a.out, "--id", iid],
                                   capture_output=True, text=True, timeout=cfg["instance_timeout_s"])
                status = "ok" if p.returncode == 0 else f"FAILED rc={p.returncode}"
                if p.returncode != 0 and not os.path.exists(os.path.join(a.out, "parts", iid + ".err")):
                    os.makedirs(os.path.join(a.out, "parts"), exist_ok=True)
                    open(os.path.join(a.out, "parts", iid + ".err"), "w").write(p.stderr[-5000:])
            except subprocess.TimeoutExpired:
                status = "TIMEOUT"
                open(os.path.join(a.out, "parts", iid + ".err"), "w").write("timeout")
            return iid, status, time.time() - t0

        with ThreadPoolExecutor(a.workers) as ex:
            for n, (iid, status, dt) in enumerate(ex.map(job, todo), 1):
                el = time.time() - t_start
                print(f"[{n}/{len(todo)}] {iid}: {status} ({dt:.0f}s) | ETA {el / n * (len(todo) - n) / 60:.0f} min",
                      flush=True)
        merge(a.out)

    elif a.cmd == "merge":
        merge(a.out)


def merge(out):
    parts = sorted(f for f in os.listdir(os.path.join(out, "parts")) if f.endswith(".csv")) \
        if os.path.isdir(os.path.join(out, "parts")) else []
    with open(os.path.join(out, "runs.csv"), "w", newline="") as fo:
        wr = csv.DictWriter(fo, fieldnames=COLUMNS)
        wr.writeheader()
        for f in parts:
            wr.writerows(csv.DictReader(open(os.path.join(out, "parts", f))))
    errs = [f for f in os.listdir(os.path.join(out, "parts")) if f.endswith(".err")] if parts else []
    print(f"merged {len(parts)} instances -> {out}/runs.csv ({len(errs)} with errors)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/bench/bench_figures.py
"""R6 figures from benchmark geometry dumps (no Fields2Cover needed).

    python bench_figures.py RESULTS/geojson/<instance_id>.geojson --out FIGDIR
writes fig_paths.pdf, fig_coverage.pdf, fig_detour_sc.pdf (174 mm wide, sans-serif 8 pt,
fonts embedded as TrueType/Type 42, RGB, coverage maps rasterised inside the vector PDF).
"""
import argparse
import json
import os
import sys

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.geometry import shape, LineString
from shapely.plotting import plot_polygon

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from nb_geom import Track, simulate_coverage  # noqa: E402

MM = 1 / 25.4
FULL_W = 174 * MM
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.5, "savefig.dpi": 600,
})
C_SW, C_TN, C_OBS, C_ZONE = "#2166ac", "#e08214", "#4d4d4d", "#bababa"
CMAP = ListedColormap(["#d73027", "#a6d96a", "#fdae61"])
STRAT = ["A", "B1", "B2", "B3"]
LABEL = {"A": "A (open-paddock plan)", "B1": "B1 holes", "B2": "B2 decomposition", "B3": "B3 drive-around"}


def load(path):
    gj = json.load(open(path))
    d = {"width": gj.get("properties", {}).get("width"), "obstacles": [], "paths": {}, "extra": {}}
    for f in gj["features"]:
        p, g = f["properties"], shape(f["geometry"])
        if p["layer"] == "paddock":
            d["field"] = g
        elif p["layer"] == "gate":
            d["gate"] = (g.x, g.y)
        elif p["layer"] == "obstacle":
            d["obstacles"].append((p["kind"], g))
        elif p["layer"] == "zone":
            d["zones"] = g
        elif p["layer"] == "path":
            d["paths"].setdefault(p["strategy"], []).append((p["seq"], g, p["kind"]))
        elif p["layer"] == "headland_pass":
            d["extra"].setdefault(p["strategy"], []).append(g)
    d["tracks"] = {s: Track([(g, k) for _, g, k in sorted(v, key=lambda t: t[0])]) for s, v in d["paths"].items()}
    npz = path.replace(".geojson", "_coverage.npz")
    d["cov"] = dict(np.load(npz)) if os.path.exists(npz) else {}
    return d


def base_ax(ax, d, zones=True):
    plot_polygon(d["field"], ax=ax, add_points=False, facecolor="#f7f7f7", edgecolor="k", linewidth=0.6)
    if zones and d.get("zones") is not None:
        plot_polygon(d["zones"], ax=ax, add_points=False, facecolor="none", edgecolor=C_ZONE, linewidth=0.4,
                     linestyle="--")
    for kind, g in d["obstacles"]:
        plot_polygon(g, ax=ax, add_points=False, facecolor="#1b7837" if kind == "tree" else "#8c6d31",
                     edgecolor="none", linewidth=0, zorder=4)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


def draw_track(ax, tr, lw=0.45):
    segs = [np.asarray(g.coords) for g, _ in tr.segs]
    cols = [C_SW if k == "swath" else C_TN for _, k in tr.segs]
    ax.add_collection(LineCollection(segs, colors=cols, linewidths=lw, zorder=5))


def fig_paths(d, out):
    strats = [s for s in STRAT if s in d["tracks"]]
    n = len(strats)
    cols = 2 if n > 2 else n
    rows = int(np.ceil(n / cols))
    fw, fh = d["field"].bounds[2] - d["field"].bounds[0], d["field"].bounds[3] - d["field"].bounds[1]
    fig, axs = plt.subplots(rows, cols, figsize=(FULL_W, FULL_W / cols * fh / fw * rows + 0.35), squeeze=False)
    for ax, s in zip(axs.ravel(), strats):
        base_ax(ax, d)
        draw_track(ax, d["tracks"][s])
        ax.plot(*d["gate"], marker="*", ms=7, mfc="gold", mec="k", mew=0.4)
        ax.set_title(f"({'abcd'[strats.index(s)]}) {LABEL[s]}", loc="left")
    for ax in axs.ravel()[n:]:
        ax.axis("off")
    fig.legend(handles=[Line2D([], [], color=C_SW, lw=1, label="working"), Line2D([], [], color=C_TN, lw=1, label="turn / headland"),
                        Patch(color="#1b7837", label="tree"), Patch(color="#8c6d31", label="rock outcrop"),
                        Line2D([], [], color=C_ZONE, ls="--", lw=0.8, label="safety zone"),
                        Line2D([], [], marker="*", ls="", mfc="gold", mec="k", label="gate")],
               loc="lower center", ncol=6, frameon=False, bbox_to_anchor=(0.5, 0))
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(out); plt.close(fig)


def cov_map(ax, d, key, field_mask=True):
    cnt = d["cov"][key].astype(float)
    ext = d["cov"]["extent"]
    sim_mask = _target_mask(d, cnt.shape, ext)
    im = ax.imshow(np.ma.masked_where(~sim_mask, np.minimum(cnt, 2)), origin="lower", extent=ext, cmap=CMAP,
                   vmin=-0.5, vmax=2.5, interpolation="nearest", rasterized=True, zorder=2)
    return im


_MASKS = {}


def _target_mask(d, shp, ext):
    import shapely
    key = (id(d), shp)
    if key not in _MASKS:
        ny, nx = shp
        xs = np.linspace(ext[0], ext[1], nx, endpoint=False) + (ext[1] - ext[0]) / nx / 2
        ys = np.linspace(ext[2], ext[3], ny, endpoint=False) + (ext[3] - ext[2]) / ny / 2
        X, Y = np.meshgrid(xs, ys)
        m = shapely.contains_xy(d["field"], X, Y)
        if d.get("zones") is not None:
            m &= ~shapely.contains_xy(d["zones"], X, Y)
        _MASKS[key] = m
    return _MASKS[key]


def fig_coverage(d, out):
    strats = [s for s in STRAT if f"{s}_sc0" in d["cov"] and (s != "A" or len(d["cov"]) <= 3)]
    n = len(strats)
    fw, fh = d["field"].bounds[2] - d["field"].bounds[0], d["field"].bounds[3] - d["field"].bounds[1]
    fig, axs = plt.subplots(2, n, figsize=(FULL_W, FULL_W / n * fh / fw * 2 + 0.45), squeeze=False)
    for j, s in enumerate(strats):
        for i, sc in enumerate((0, 1)):
            ax = axs[i, j]
            base_ax(ax, d, zones=False)
            cov_map(ax, d, f"{s}_sc{sc}")
            cnt, m = d["cov"][f"{s}_sc{sc}"], _target_mask(d, d["cov"][f"{s}_sc{sc}"].shape, d["cov"]["extent"])
            ax.set_title(f"{s} · {'SC on' if sc else 'SC off'}\nmissed {100 * (m & (cnt == 0)).sum() / m.sum():.1f}% · "
                         f"overlap {100 * (m & (cnt >= 2)).sum() / m.sum():.1f}%", loc="left")
    fig.legend(handles=[Patch(color=CMAP(0), label="missed (0)"), Patch(color=CMAP(1), label="once (1)"),
                        Patch(color=CMAP(2), label="overlap (≥2)")], loc="lower center", ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(out); plt.close(fig)


def fig_detour(d, out, n_sections=None, step=2.0):
    """B3 detour zoom: coverage without SC (left) and section ON/OFF footprints with SC (right)."""
    tr, w = d["tracks"]["B3"], float(d["width"])
    n_sections = n_sections or {12: 6, 24: 12, 36: 18}.get(int(w), 6)
    # obstacle whose safety zone is crossed by the most working pieces
    best = max(d["obstacles"], key=lambda o: sum(g.distance(o[1]) < w for g in tr.pieces("swath")))[1]
    win = best.buffer(2.2 * w).envelope
    sims = {sc: simulate_coverage(tr, d["field"], d.get("zones"), w, n_sections, 1.0, section_control=sc,
                                  extra_passes=d["extra"].get("B3", []), record_window=win if sc else None)
            for sc in (False, True)}
    x0, y0, x1, y1 = win.bounds
    fig, axs = plt.subplots(1, 2, figsize=(FULL_W, FULL_W / 2 * (y1 - y0) / (x1 - x0) + 0.4))
    for ax, sc in zip(axs, (False, True)):
        base_ax(ax, d)
        r = sims[sc]
        ax.imshow(np.ma.masked_where(~r["target"], np.minimum(r["count"], 2)), origin="lower",
                  extent=r["extent"], cmap=CMAP, vmin=-0.5, vmax=2.5, interpolation="nearest",
                  alpha=0.35 if sc else 1.0, rasterized=True, zorder=2)
        if sc:  # boom ribbon of the pass that switches sections most often in the window
            by_pass = {}
            for pid, x, y, dx, dy, on in r["states"]:
                lst = by_pass.setdefault(pid, [])
                if not lst or np.hypot(x - lst[-1][0], y - lst[-1][1]) >= step:
                    lst.append((x, y, dx, dy, on))
            switches = {pid: sum(int((a[4] != b[4]).sum()) for a, b in zip(s[:-1], s[1:])) for pid, s in by_pass.items()}
            pid = max(switches, key=switches.get)
            on_q, off_q, edges = [], [], np.linspace(-w / 2, w / 2, n_sections + 1)
            s = by_pass[pid]
            for (x, y, dx, dy, on), (x2, y2, dx2, dy2, _) in zip(s[:-1], s[1:]):
                c, c2 = np.array([x, y]), np.array([x2, y2])
                n1, n2 = np.array([-dy, dx]), np.array([-dy2, dx2])
                for j in range(n_sections):
                    a, b = edges[j], edges[j + 1]
                    (on_q if on[j] else off_q).append([c + a * n1, c + b * n1, c2 + b * n2, c2 + a * n2])
            ax.add_collection(PolyCollection(off_q, facecolors="#bdbdbd", edgecolors="none", rasterized=True, zorder=3))
            ax.add_collection(PolyCollection(on_q, facecolors="#4393c3", edgecolors="none", rasterized=True, zorder=3))
            # section dividers
            for j in range(1, n_sections):
                pts = np.array([(x - edges[j] * dy, y + edges[j] * dx) for x, y, dx, dy, _ in s])
                ax.plot(pts[:, 0], pts[:, 1], color="white", lw=0.25, zorder=3.5)
        draw_track(ax, tr, lw=0.8)
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
        ax.set_title("(a) SC off: coverage count" if not sc else f"(b) SC on: coverage + {n_sections}-section ON/OFF of one pass", loc="left")
    fig.legend(handles=[Patch(color="#4393c3", label="section ON (one pass)"), Patch(color="#bdbdbd", label="section OFF"),
                        Patch(color=CMAP(0), label="missed"), Patch(color=CMAP(1), label="once"),
                        Patch(color=CMAP(2), label="overlap"), Line2D([], [], color=C_SW, label="B3 path")],
               loc="lower center", ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.savefig(out); plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("geojson")
    ap.add_argument("--out", default=".")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    d = load(a.geojson)
    fig_paths(d, os.path.join(a.out, "fig_paths.pdf"))
    fig_coverage(d, os.path.join(a.out, "fig_coverage.pdf"))
    if "B3" in d["tracks"]:
        fig_detour(d, os.path.join(a.out, "fig_detour_sc.pdf"))
    print("figures ->", a.out)


if __name__ == "__main__":
    main()

## Design, pilot, full run
`design` writes `design.csv` and `run_meta.json`. The meta file holds the full CONFIG (including the cost weights c_t, c_p and c_m), the package
versions and the source hashes, which makes the run reproducible. The CONFIG is read from `run_meta.json` on every resume, so **do not edit it mid-run**.

In [ ]:
import json
if not os.path.exists(f"{OUT}/design.csv"):
    subprocess.run(["python", "f2c_bench.py", "design", "--out", OUT], check=True)
meta = json.load(open(f"{OUT}/run_meta.json"))
print({k: meta[k] for k in ("bench_version", "fields2cover", "created")}, "| costs:",
      {k: meta["config"][k] for k in ("c_t", "c_p", "c_m")})

In [ ]:
# Pilot: 4 instances (about 3-5 min). Check runs.csv before starting the full run.
!python f2c_bench.py run --out "{OUT}" --workers 2 --limit 4 --filter "_s0"
import pandas as pd
df = pd.read_csv(f"{OUT}/runs.csv"); display(df.head(12)); print(df.shape)
!ls "{OUT}/parts" | grep err || echo "no errors"

In [ ]:
# Full run (resumable). If Colab disconnects, re-run cells 1, 2, the %%writefile cells and this one.
!python f2c_bench.py run --out "{OUT}" --workers 2

In [ ]:
# Merge + quick summary
!python f2c_bench.py merge --out "{OUT}"
df = pd.read_csv(f"{OUT}/runs.csv")
print(df.shape, "instances:", df.instance_id.nunique())
display(df[df.sc == 1].groupby(["strategy"])[["time_min", "missed_pct", "overlap_pct", "product_saved_pct", "crit_C"]].median())

In [ ]:
# R6 figures for one illustrative instance (vector PDF, 174 mm, embedded fonts)
inst = "rectangle_mixed_thomas_d1_w24_road_s0"
!python bench_figures.py "{OUT}/geojson/{inst}.geojson" --out "{OUT}/figures/{inst}"
!ls "{OUT}/figures/{inst}"

## Headless on a Linux VM (e.g. Nectar)
```bash
bash docs/nectar_setup.sh                       # builds OR-Tools + F2C into a venv
source ~/f2c-venv/bin/activate                  # (the path printed by the setup script)
cd tools
python f2c_bench.py design --out ../results/benchmark_v1
nohup python f2c_bench.py run --out ../results/benchmark_v1 --workers $(nproc) > ../results/benchmark_v1/run.log 2>&1 &
python bench_figures.py ../results/benchmark_v1/geojson/<instance>.geojson --out ../results/benchmark_v1/figures
```